# Sonification Mapping: Vp to Pitch

## Current status

This notebook currently tests duration-aware pitch transitions for solar-wind speed (`Vp`) after the earlier continuous, discrete-event, sustained-state, narrow-range, and Clip G MIDI experiments. Those completed experiments and listening notes are summarized in `../docs/experiments/vp_pitch_experiments.md`.

The present output remains a set of reproducible `Vp -> pitch` comparison clips, plots, lookup tables, and the optional Clip G MIDI export. The unresolved question is how to make transitions between sustained pitch states easier to follow without hiding short-lived `Vp` changes or crossing missing-data gaps.

Notebook 03 established the cleaning rules and first-pass normalization decisions for ACE SWEPAM variables. This notebook keeps the scope narrow: only `Vp` is mapped to pitch, while missing data remain silent and amplitude stays data-independent.


## Design Principles

HelioSonic's current mapping is guided by the following principles:

- Preserve scientific relationships: higher or lower data values should have consistent auditory meaning unless an inversion is explicitly documented.
- Preserve temporal structure: gaps, cadence, and relative changes over time should not be hidden by the mapping process.
- Use perceptually distinguishable auditory dimensions: pitch, loudness, and timbre should be reserved for differences listeners can plausibly separate.
- Avoid misleading artifacts: mapping choices should not imply precision, continuity, or periodicity that is not present in the spacecraft data.
- Separate data processing from audio synthesis: cleaning, normalization, mapping, control-signal smoothing, and sound generation should remain distinct steps.
- Prioritize interpretability over musicality: the first question is what the sound communicates about the measurement, not whether it sounds composed.

These are not the only possible sonification principles; they are the present design commitments for this research phase of HelioSonic.

## Human Listening Feedback — First Prototype

The following notes are subjective qualitative listening observations from the first 12-second sine-wave prototype. They are not a perceptual evaluation experiment and should not be interpreted as general listener results.

- The 12-second playback was too fast for detailed analysis.
- The broad pitch contour was difficult to follow and sounded somewhat random or noise-like.
- A larger missing-data gap was audible near seven seconds.
- Smaller gaps may have contributed to a busy or fragmented sound.
- Audible pops, buzzing, or gain-like peaks were present.
- The 300-600 Hz logarithmic mapping was more comfortable and wind-like but had less perceptible pitch variation.
- The 220-880 Hz logarithmic mapping produced clearer pitch contrast than the linear version.
- The inverted logarithmic version was recognizable as reversing the pitch direction.
- None of the current outputs were sufficiently comfortable or interpretable for scientific listening.

## Revised Research Question

Under what playback-duration and pitch-control smoothing settings does the `Vp` contour become easier to follow without erasing scientifically meaningful changes?

This revision uses one baseline mapping only: non-inverted 220-880 Hz logarithmic pitch mapping. The experiment varies playback duration, segment-aware pitch-control smoothing, and one optional phase-reset strategy at valid-segment starts. It does not test new pitch ranges, inversion, amplitude mappings, instruments, MIDI, reverb, or musical effects.

## Load and Prepare Vp

The loading and cleaning logic below follows Notebook 03: CDF fill values and valid physical ranges are used to identify valid samples, invalid samples are preserved as `NaN`, and the original time axis is retained. For `Vp`, Notebook 03 selected min-max normalization as the first candidate for sonification because the variable is physically meaningful and bounded, while also noting that the observed speed range is narrow in this file.

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import cdflib
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio, Markdown, display

DATA_PATH = Path("../data/ac_h0_swe_20240709_v11.cdf")
VARIABLE = "Vp"
VARIABLE_LABEL = "Solar wind speed"

cdf = cdflib.CDF(DATA_PATH)
times = cdflib.cdfepoch.to_datetime(cdf.varget("Epoch"))
time64 = np.asarray(times, dtype="datetime64[ns]")
source_offsets_seconds = ((time64 - time64[0]) / np.timedelta64(1, "s")).astype(float)
source_cadence_seconds = float(np.nanmedian(np.diff(source_offsets_seconds)))


def format_table_value(value):
    """Format values for compact Markdown table display."""
    if isinstance(value, float):
        return f"{value:.3f}"
    return str(value)


def display_markdown_table(rows, columns=None, max_rows=None):
    """Display a list of dictionaries as a Markdown table without extra dependencies."""
    if max_rows is not None:
        rows = rows[:max_rows]
    if not rows:
        display(Markdown("_No rows to display._"))
        return
    if columns is None:
        columns = list(rows[0].keys())

    header = "| " + " | ".join(columns) + " |"
    separator = "| " + " | ".join(["---"] * len(columns)) + " |"
    body = [
        "| " + " | ".join(format_table_value(row.get(column, "")) for column in columns) + " |"
        for row in rows
    ]
    display(Markdown("\n".join([header, separator] + body)))


print(f"Loaded: {DATA_PATH.name}")
print(f"Samples: {len(times)}")
print(f"Approximate source cadence: {source_cadence_seconds:.1f} seconds")

In [ ]:
def clean_variable(cdf_file, variable_name, time_values):
    """Load one CDF variable and identify samples that are scientifically valid."""
    values = np.asarray(cdf_file.varget(variable_name), dtype=float)
    attrs = cdf_file.varattsget(variable_name)

    mask = np.isfinite(values)

    fill_value = attrs.get("FILLVAL")
    if fill_value is not None:
        mask &= values != fill_value

    valid_min = attrs.get("VALIDMIN")
    if valid_min is not None:
        mask &= values >= valid_min

    valid_max = attrs.get("VALIDMAX")
    if valid_max is not None:
        mask &= values <= valid_max

    return {
        "name": variable_name,
        "label": attrs.get("FIELDNAM", variable_name),
        "description": attrs.get("CATDESC", ""),
        "units": attrs.get("UNITS", ""),
        "values": values,
        "times": time_values,
        "time64": time64,
        "source_offsets_seconds": source_offsets_seconds,
        "mask": mask,
        "valid_values": values[mask],
        "attrs": attrs,
    }


def apply_mask_as_nan(values, mask):
    """Preserve the original time axis while marking invalid samples as NaN."""
    cleaned = np.asarray(values, dtype=float).copy()
    cleaned[~mask] = np.nan
    return cleaned


def min_max_normalize(values):
    """Normalize finite values to 0-1 while preserving NaN gaps."""
    values = np.asarray(values, dtype=float)
    finite_mask = np.isfinite(values)
    normalized = np.full_like(values, np.nan, dtype=float)

    if not np.any(finite_mask):
        return normalized

    data_min = np.nanmin(values)
    data_max = np.nanmax(values)

    if data_max == data_min:
        normalized[finite_mask] = 0.0
        return normalized

    normalized[finite_mask] = (values[finite_mask] - data_min) / (data_max - data_min)
    return normalized


vp_record = clean_variable(cdf, VARIABLE, times)
vp_record["nan_values"] = apply_mask_as_nan(vp_record["values"], vp_record["mask"])
vp_record["normalized"] = min_max_normalize(vp_record["nan_values"])

valid_count = int(np.sum(vp_record["mask"]))
invalid_count = len(vp_record["values"]) - valid_count

print(f"Variable: {VARIABLE} - {VARIABLE_LABEL}")
print(f"Units: {vp_record['units']}")
print(f"Valid samples: {valid_count}")
print(f"Invalid samples: {invalid_count} ({100 * invalid_count / len(vp_record['values']):.2f}%)")
print(f"Observed valid range: {np.nanmin(vp_record['nan_values']):.2f} to {np.nanmax(vp_record['nan_values']):.2f} {vp_record['units']}")

## Mapping Configuration

This revision uses one baseline pitch mapping: non-inverted 220-880 Hz logarithmic mapping. The earlier listening feedback suggested this mapping provided clearer pitch contrast than the linear version, while avoiding the direction reversal of the inverted version. New pitch ranges are intentionally not tested here.

In [ ]:
@dataclass(frozen=True)
class PitchMappingConfig:
    """Parameters for mapping normalized data values to pitch in Hz."""

    min_frequency_hz: float = 220.0
    max_frequency_hz: float = 880.0
    scale: str = "log"
    invert: bool = False
    label: str = "220-880 Hz log baseline"


baseline_pitch_config = PitchMappingConfig()
baseline_pitch_config

## Mapping Function

The pitch mapping function accepts normalized values and returns frequency values in Hz. It preserves `NaN` samples so later synthesis can represent invalid or missing spacecraft samples as silence rather than interpolated pitch.

In [ ]:
def normalized_to_pitch(normalized_values, config):
    """Map normalized 0-1 values to pitch frequencies in Hz.

    Parameters
    ----------
    normalized_values : array-like
        Values expected to be normalized to the inclusive 0-1 range. NaN values
        are preserved as NaN so invalid data intervals remain explicit.
    config : PitchMappingConfig
        Mapping parameters controlling frequency range, scale type, and inversion.

    Returns
    -------
    numpy.ndarray
        Pitch frequencies in Hz, with NaN values preserved.
    """
    if not isinstance(config, PitchMappingConfig):
        raise TypeError("config must be a PitchMappingConfig")
    if config.min_frequency_hz <= 0 or config.max_frequency_hz <= 0:
        raise ValueError("Frequencies must be positive for pitch mapping")
    if config.max_frequency_hz <= config.min_frequency_hz:
        raise ValueError("max_frequency_hz must be greater than min_frequency_hz")
    if config.scale not in {"linear", "log"}:
        raise ValueError("scale must be either 'linear' or 'log'")

    values = np.asarray(normalized_values, dtype=float)
    finite_mask = np.isfinite(values)

    if np.any((values[finite_mask] < 0) | (values[finite_mask] > 1)):
        raise ValueError("Finite normalized values must be between 0 and 1")

    mapped_position = values.copy()
    if config.invert:
        mapped_position[finite_mask] = 1.0 - mapped_position[finite_mask]

    pitch = np.full_like(values, np.nan, dtype=float)
    if config.scale == "linear":
        pitch[finite_mask] = config.min_frequency_hz + mapped_position[finite_mask] * (
            config.max_frequency_hz - config.min_frequency_hz
        )
    else:
        frequency_ratio = config.max_frequency_hz / config.min_frequency_hz
        pitch[finite_mask] = config.min_frequency_hz * (frequency_ratio ** mapped_position[finite_mask])

    return pitch


vp_pitch_hz = normalized_to_pitch(vp_record["normalized"], baseline_pitch_config)

## Segment Diagnostics

Before modifying the auditory control path, inspect the valid and missing-data structure. These diagnostics use the original sample mask and timestamps. They do not alter the spacecraft measurements.

In [ ]:
def find_boolean_segments(mask, value=True):
    """Return inclusive index ranges where a boolean mask equals value."""
    mask = np.asarray(mask, dtype=bool)
    target = mask == value
    segments = []
    start = None

    for index, is_target in enumerate(target):
        if is_target and start is None:
            start = index
        elif not is_target and start is not None:
            segments.append((start, index - 1))
            start = None

    if start is not None:
        segments.append((start, len(mask) - 1))

    return segments


def summarize_segments(segments, segment_type, cadence_seconds):
    rows = []
    for start, end in segments:
        sample_count = end - start + 1
        rows.append({
            "type": segment_type,
            "start_index": start,
            "end_index": end,
            "sample_count": sample_count,
            "duration_seconds": sample_count * cadence_seconds,
            "start_time": vp_record["times"][start],
            "end_time": vp_record["times"][end],
        })
    return rows


valid_segments = find_boolean_segments(vp_record["mask"], value=True)
missing_segments = find_boolean_segments(vp_record["mask"], value=False)
valid_segment_rows = summarize_segments(valid_segments, "valid", source_cadence_seconds)
missing_gap_rows = summarize_segments(missing_segments, "missing", source_cadence_seconds)

short_segment_threshold_samples = 2
short_valid_count = sum(row["sample_count"] <= short_segment_threshold_samples for row in valid_segment_rows)
short_missing_count = sum(row["sample_count"] <= short_segment_threshold_samples for row in missing_gap_rows)

print(f"Valid segments: {len(valid_segments)}")
print(f"Missing-data gaps: {len(missing_segments)}")
print(f"Very short valid segments (<= {short_segment_threshold_samples} samples): {short_valid_count}")
print(f"Very short missing segments (<= {short_segment_threshold_samples} samples): {short_missing_count}")

largest_missing_gaps = sorted(missing_gap_rows, key=lambda row: row["duration_seconds"], reverse=True)
display_markdown_table(largest_missing_gaps, max_rows=10)

## Pitch-Control Smoothing

The auditory path in this experiment is:

`cleaned Vp -> min-max normalization -> logarithmic pitch mapping -> pitch-control smoothing -> synthesis`

Smoothing is applied only to the pitch-control signal used by the synthesizer. It does not alter or overwrite the original `Vp` values, timestamps, validity mask, normalized values, or unsmoothed mapped pitch. The original source timestamps and values remain available in `vp_record` for later lookup.

The smoothing method is a centered moving average with configurable source-sample window size. It operates separately inside each continuous valid-data segment, preserves `NaN` gaps, never averages across a missing-data interval, and never fills missing values.

In [ ]:
@dataclass(frozen=True)
class PitchSmoothingConfig:
    """Segment-aware pitch-control smoothing configuration."""

    label: str
    window_samples: int


smoothing_configs = {
    "none": PitchSmoothingConfig("no smoothing", 1),
    "light": PitchSmoothingConfig("light smoothing", 5),
    "moderate": PitchSmoothingConfig("moderate smoothing", 15),
}


def smooth_pitch_by_valid_segment(pitch_hz, valid_mask, smoothing_config):
    """Smooth pitch within valid segments while preserving NaN gaps."""
    if not isinstance(smoothing_config, PitchSmoothingConfig):
        raise TypeError("smoothing_config must be a PitchSmoothingConfig")
    if smoothing_config.window_samples < 1:
        raise ValueError("window_samples must be at least 1")

    pitch_hz = np.asarray(pitch_hz, dtype=float)
    valid_mask = np.asarray(valid_mask, dtype=bool) & np.isfinite(pitch_hz)
    smoothed = np.full_like(pitch_hz, np.nan, dtype=float)

    for start, end in find_boolean_segments(valid_mask, value=True):
        segment = pitch_hz[start:end + 1]
        if smoothing_config.window_samples == 1 or len(segment) == 1:
            smoothed[start:end + 1] = segment
            continue

        window = min(smoothing_config.window_samples, len(segment))
        kernel = np.ones(window, dtype=float)
        numerator = np.convolve(segment, kernel, mode="same")
        denominator = np.convolve(np.ones_like(segment), kernel, mode="same")
        smoothed[start:end + 1] = numerator / denominator

    return smoothed


pitch_controls = {
    key: smooth_pitch_by_valid_segment(vp_pitch_hz, vp_record["mask"], config)
    for key, config in smoothing_configs.items()
}

for key, config in smoothing_configs.items():
    print(f"{key}: {config.label}, window={config.window_samples} source samples")

## Interpolated Synthesizer Control

The original ACE SWEPAM timestamps and values are preserved. Interpolation here affects only the synthesizer control trajectory between measured samples after mapping to pitch. Linear interpolation is performed separately within each continuous valid-data segment. It is not performed across missing-data gaps, and missing samples remain silent.

In [ ]:
def interpolate_pitch_control_to_audio_grid(pitch_hz, valid_mask, output_duration_seconds, sample_rate):
    """Interpolate pitch control within valid source segments onto an audio grid."""
    pitch_hz = np.asarray(pitch_hz, dtype=float)
    valid_mask = np.asarray(valid_mask, dtype=bool) & np.isfinite(pitch_hz)

    if output_duration_seconds <= 0:
        raise ValueError("output_duration_seconds must be positive")
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive")

    output_sample_count = int(round(output_duration_seconds * sample_rate))
    source_positions = np.linspace(0, len(pitch_hz) - 1, output_sample_count)
    interpolated_pitch = np.full(output_sample_count, np.nan, dtype=float)
    source_indices = np.arange(len(pitch_hz), dtype=float)

    for start, end in find_boolean_segments(valid_mask, value=True):
        audio_segment_mask = (source_positions >= start) & (source_positions <= end)
        if not np.any(audio_segment_mask):
            continue
        if start == end:
            interpolated_pitch[audio_segment_mask] = pitch_hz[start]
            continue

        interpolated_pitch[audio_segment_mask] = np.interp(
            source_positions[audio_segment_mask],
            source_indices[start:end + 1],
            pitch_hz[start:end + 1],
        )

    return interpolated_pitch


example_interpolated_pitch = interpolate_pitch_control_to_audio_grid(
    pitch_controls["light"], vp_record["mask"], output_duration_seconds=60, sample_rate=22050
)

print("Interpolated 60-second control samples:", len(example_interpolated_pitch))
print("Silent control samples:", int(np.sum(~np.isfinite(example_interpolated_pitch))))

## Synthesis Method

The synthesizer remains intentionally simple: a single sine wave, constant amplitude during valid data, and exact silence during missing data. Amplitude is not varied according to `Vp`.

Amplitude fades were intentionally excluded because loudness may later encode a separate scientific variable. Gap handling must therefore be distinguishable from data-driven amplitude changes. This means clicks at gap boundaries are not hidden in this experiment; they are diagnosed directly.

In [ ]:
def synthesize_sine_from_interpolated_pitch(
    interpolated_pitch_hz,
    sample_rate=22050,
    amplitude=0.2,
    phase_behavior="continuous",
):
    """Synthesize a constant-amplitude sine wave from interpolated pitch control.

    NaN pitch samples become exact silence. With ``phase_behavior='reset_at_valid_segment_start'``,
    each valid segment begins at oscillator phase zero. This can reduce entry clicks,
    but it may also make separated data intervals sound like restarted events.
    """
    if phase_behavior not in {"continuous", "reset_at_valid_segment_start"}:
        raise ValueError("Unsupported phase_behavior")
    if sample_rate <= 0:
        raise ValueError("sample_rate must be positive")
    if amplitude < 0 or amplitude > 1:
        raise ValueError("amplitude must be between 0 and 1")

    pitch = np.asarray(interpolated_pitch_hz, dtype=float)
    valid_audio_mask = np.isfinite(pitch)
    audio = np.zeros_like(pitch, dtype=float)

    if phase_behavior == "continuous":
        safe_pitch = np.where(valid_audio_mask, pitch, 0.0)
        phase = 2 * np.pi * np.cumsum(safe_pitch) / sample_rate
        audio[valid_audio_mask] = amplitude * np.sin(phase[valid_audio_mask])
        return audio, valid_audio_mask

    for start, end in find_boolean_segments(valid_audio_mask, value=True):
        segment_pitch = pitch[start:end + 1]
        segment_phase = 2 * np.pi * np.cumsum(segment_pitch) / sample_rate
        audio[start:end + 1] = amplitude * np.sin(segment_phase)

    return audio, valid_audio_mask


sample_rate = 22050
constant_amplitude = 0.2

## Click and Transition Diagnostics

The first listening pass reported pops, buzzing, or gain-like peaks. Likely causes include discontinuous waveform values when silence begins or ends, abrupt frequency-control changes, oscillator phase behavior at segment boundaries, and dense sequences of short valid or invalid intervals.

The diagnostics below identify segment counts, missing-gap durations, short segments, waveform values immediately before and after major audio transitions, peak amplitude, and whether clipping occurred. The purpose is to understand the artifacts without hiding them using fades.

In [ ]:
def audio_transition_diagnostics(audio, valid_audio_mask, sample_rate, top_n=12):
    """Summarize waveform jumps at transitions between sound and silence."""
    audio = np.asarray(audio, dtype=float)
    valid_audio_mask = np.asarray(valid_audio_mask, dtype=bool)
    transition_indices = np.flatnonzero(valid_audio_mask[1:] != valid_audio_mask[:-1]) + 1

    rows = []
    for index in transition_indices:
        before_index = index - 1
        after_index = index
        rows.append({
            "time_seconds": index / sample_rate,
            "transition": "valid_start" if valid_audio_mask[after_index] else "gap_start",
            "sample_before": audio[before_index],
            "sample_after": audio[after_index],
            "absolute_jump": abs(audio[after_index] - audio[before_index]),
        })

    return sorted(rows, key=lambda row: row["absolute_jump"], reverse=True)[:top_n]


def summarize_audio_clip(audio, valid_audio_mask):
    peak_amplitude = float(np.max(np.abs(audio))) if len(audio) else 0.0
    return {
        "peak_amplitude": peak_amplitude,
        "clipping_occurred": bool(np.any(np.abs(audio) >= 1.0)),
        "audio_valid_segments": len(find_boolean_segments(valid_audio_mask, value=True)),
        "audio_silent_segments": len(find_boolean_segments(valid_audio_mask, value=False)),
    }


diagnostic_pitch = interpolate_pitch_control_to_audio_grid(pitch_controls["light"], vp_record["mask"], 60, sample_rate)
diagnostic_audio, diagnostic_valid_mask = synthesize_sine_from_interpolated_pitch(
    diagnostic_pitch,
    sample_rate=sample_rate,
    amplitude=constant_amplitude,
    phase_behavior="continuous",
)

print(summarize_audio_clip(diagnostic_audio, diagnostic_valid_mask))
display_markdown_table(audio_transition_diagnostics(diagnostic_audio, diagnostic_valid_mask, sample_rate))

## Comparison Plots

The 60-second plots compare the original normalized `Vp`, the unsmoothed mapped pitch, lightly smoothed pitch, moderately smoothed pitch, and the validity mask. Smoothing is shown only as an auditory control-signal transformation; the normalized data trace remains the source reference.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 11), sharex=True)

axes[0].plot(vp_record["times"], vp_record["normalized"], linewidth=1, color="black")
axes[0].set_title("Original normalized Vp (source reference)")
axes[0].set_ylabel("0-1")
axes[0].grid(alpha=0.3)

axes[1].plot(vp_record["times"], pitch_controls["none"], linewidth=1)
axes[1].set_title("Unsmoothed mapped pitch control")
axes[1].set_ylabel("Hz")
axes[1].grid(alpha=0.3)

axes[2].plot(vp_record["times"], pitch_controls["light"], linewidth=1)
axes[2].set_title("Lightly smoothed pitch control (5 source samples)")
axes[2].set_ylabel("Hz")
axes[2].grid(alpha=0.3)

axes[3].plot(vp_record["times"], pitch_controls["moderate"], linewidth=1)
axes[3].set_title("Moderately smoothed pitch control (15 source samples)")
axes[3].set_ylabel("Hz")
axes[3].grid(alpha=0.3)

axes[4].step(vp_record["times"], vp_record["mask"].astype(int), where="post", color="tab:red")
axes[4].set_title("Validity mask: 1 = valid data, 0 = missing or invalid")
axes[4].set_ylabel("valid")
axes[4].set_xlabel("Time")
axes[4].set_ylim(-0.1, 1.1)
axes[4].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Controlled Audio Comparison Set

The clips below use constant amplitude for all valid data and exact silence for invalid data. They use interpolated pitch control within valid segments and hard gaps at missing samples. The sixth diagnostic clip tests whether resetting oscillator phase to zero at each valid-segment start reduces entry clicks without introducing amplitude fades.

In [ ]:
@dataclass(frozen=True)
class ClipConfig:
    label: str
    duration_seconds: int
    smoothing_key: str
    gap_behavior: str = "hard gaps"
    phase_behavior: str = "continuous"


clip_configs = [
    ClipConfig("60 seconds - no smoothing - interpolated pitch - hard gaps", 60, "none"),
    ClipConfig("60 seconds - light smoothing - interpolated pitch - hard gaps", 60, "light"),
    ClipConfig("60 seconds - moderate smoothing - interpolated pitch - hard gaps", 60, "moderate"),
    ClipConfig("30 seconds - light smoothing - interpolated pitch - hard gaps", 30, "light"),
    ClipConfig("120 seconds - light smoothing - interpolated pitch - hard gaps", 120, "light"),
    ClipConfig(
        "60 seconds - light smoothing - interpolated pitch - hard gaps - phase reset at valid-segment starts",
        60,
        "light",
        phase_behavior="reset_at_valid_segment_start",
    ),
]

clip_outputs = {}
clip_diagnostics = {}

for clip_config in clip_configs:
    interpolated_pitch = interpolate_pitch_control_to_audio_grid(
        pitch_controls[clip_config.smoothing_key],
        vp_record["mask"],
        output_duration_seconds=clip_config.duration_seconds,
        sample_rate=sample_rate,
    )
    audio, valid_audio_mask = synthesize_sine_from_interpolated_pitch(
        interpolated_pitch,
        sample_rate=sample_rate,
        amplitude=constant_amplitude,
        phase_behavior=clip_config.phase_behavior,
    )
    clip_outputs[clip_config.label] = audio
    clip_diagnostics[clip_config.label] = summarize_audio_clip(audio, valid_audio_mask)

for label, audio in clip_outputs.items():
    print(label)
    display(Audio(audio, rate=sample_rate))

In [ ]:
diagnostic_rows = []
for label, diagnostics in clip_diagnostics.items():
    diagnostic_rows.append({"clip": label, **diagnostics})

display_markdown_table(diagnostic_rows)

## Human Listening Feedback — Continuous Pitch Prototype

The following notes are subjective qualitative listening observations from the continuous sine-wave pitch prototype. This is a qualitative listening result, not a formal perceptual study.

- 60 seconds was a reasonable playback duration.
- Continuous sine-wave pitch sounded robotic, harsh, and unpleasant.
- All smoothing levels preserved audible variation but remained unsuitable for sustained analysis.
- Pops were still audible.
- Comparing the existing clips in finer detail is not considered useful because the entire continuous-frequency approach performed poorly perceptually.

Based on this listening pass, continuous `Vp -> frequency` synthesis should no longer be treated as the primary candidate for HelioSonic sonification.

# Discrete Pitch Event Prototype

## Research Question

Can temporal aggregation and discrete pitch events make `Vp` trends more interpretable and comfortable than continuous frequency mapping?

This experiment continues to use only `Vp` and keeps total playback duration fixed at 60 seconds. Instead of driving oscillator frequency continuously, the data are aggregated into time bins. Each sufficiently valid bin becomes one constant-amplitude pitch event. Bins without enough valid spacecraft data become silence.

## Discrete Event Design

The discrete event prototype is deliberately conservative:

- Only `Vp` is used.
- Playback duration is fixed at 60 seconds.
- Time bins are configurable by fixed note count over the spacecraft interval.
- Each bin uses the median valid `Vp` value as its representative value.
- A configurable minimum valid-data fraction determines whether a bin becomes a note or silence.
- Aggregated representative `Vp` values are min-max normalized, matching the first-pass `Vp` normalization approach documented in Notebook 03.
- Normalized values map to chromatic semitone-spaced pitches over a limited range of roughly one to one-and-a-half octaves.
- Every valid note uses the same constant amplitude.
- A short fixed attack and release envelope is applied to every note only to prevent clicks. This envelope is fixed synthesis behavior, not a data-driven loudness mapping.
- The waveform uses a simple gentle tone with modest harmonic content. No reverb, chords, accompaniment, musical key, MIDI export, or event-detection accents are added.

In [ ]:
@dataclass(frozen=True)
class DiscreteEventConfig:
    """Configuration for aggregating Vp into discrete pitch events."""

    label: str
    note_count: int
    playback_duration_seconds: float = 60.0
    minimum_valid_fraction: float = 0.5
    representative_statistic: str = "median"
    min_midi_note: int = 60
    max_midi_note: int = 76
    amplitude: float = 0.18
    attack_seconds: float = 0.015
    release_seconds: float = 0.035
    sample_rate: int = 22050


# 90 notes is included as a midpoint between one event per second and two events per second.
discrete_event_configs = [
    DiscreteEventConfig("60 notes over 60 seconds", note_count=60),
    DiscreteEventConfig("90 notes over 60 seconds", note_count=90),
    DiscreteEventConfig("120 notes over 60 seconds", note_count=120),
]

discrete_event_configs

In [ ]:
NOTE_NAMES = ["C", "C#", "D", "D#", "E", "F", "F#", "G", "G#", "A", "A#", "B"]


def midi_to_frequency(midi_note):
    """Convert a MIDI note number to equal-tempered frequency in Hz."""
    return 440.0 * (2.0 ** ((np.asarray(midi_note, dtype=float) - 69.0) / 12.0))


def midi_to_note_name(midi_note):
    """Return a chromatic note label for a MIDI note number."""
    midi_note = int(midi_note)
    octave = midi_note // 12 - 1
    return f"{NOTE_NAMES[midi_note % 12]}{octave}"


def normalized_to_midi_note(normalized_values, min_midi_note, max_midi_note):
    """Map normalized 0-1 values to rounded chromatic MIDI note numbers."""
    values = np.asarray(normalized_values, dtype=float)
    finite_mask = np.isfinite(values)
    midi_notes = np.full(values.shape, np.nan, dtype=float)

    if min_midi_note >= max_midi_note:
        raise ValueError("min_midi_note must be less than max_midi_note")
    if np.any((values[finite_mask] < 0) | (values[finite_mask] > 1)):
        raise ValueError("Finite normalized values must be between 0 and 1")

    mapped = min_midi_note + values[finite_mask] * (max_midi_note - min_midi_note)
    midi_notes[finite_mask] = np.rint(mapped)
    return midi_notes

## Temporal Aggregation

The aggregation step divides the original spacecraft time interval into fixed-count bins. Each bin stores its audio interval, source spacecraft interval, median `Vp`, valid-data fraction, normalized value, note number, note name, and frequency. The lookup table preserves the connection between each audible event and the original data interval.

A bin with insufficient valid data remains silent. Missing or rejected bins are not filled.

In [ ]:
def aggregate_vp_to_bins(record, config):
    """Aggregate cleaned Vp values into fixed-count time bins."""
    if config.note_count <= 0:
        raise ValueError("note_count must be positive")
    if not 0 <= config.minimum_valid_fraction <= 1:
        raise ValueError("minimum_valid_fraction must be between 0 and 1")
    if config.representative_statistic != "median":
        raise ValueError("Only median aggregation is implemented for this prototype")

    values = np.asarray(record["nan_values"], dtype=float)
    valid_mask = np.asarray(record["mask"], dtype=bool) & np.isfinite(values)
    source_offsets = np.asarray(record["source_offsets_seconds"], dtype=float)
    source_start = float(source_offsets[0])
    source_end = float(source_offsets[-1])
    source_edges = np.linspace(source_start, source_end, config.note_count + 1)
    audio_edges = np.linspace(0.0, config.playback_duration_seconds, config.note_count + 1)

    rows = []
    representative_values = np.full(config.note_count, np.nan, dtype=float)
    valid_fractions = np.zeros(config.note_count, dtype=float)

    for bin_index in range(config.note_count):
        left = source_edges[bin_index]
        right = source_edges[bin_index + 1]
        if bin_index == config.note_count - 1:
            in_bin = (source_offsets >= left) & (source_offsets <= right)
        else:
            in_bin = (source_offsets >= left) & (source_offsets < right)

        sample_count = int(np.sum(in_bin))
        valid_count = int(np.sum(in_bin & valid_mask))
        valid_fraction = valid_count / sample_count if sample_count else 0.0
        valid_fractions[bin_index] = valid_fraction

        if sample_count and valid_fraction >= config.minimum_valid_fraction:
            representative_values[bin_index] = float(np.nanmedian(values[in_bin & valid_mask]))

        rows.append({
            "bin_index": bin_index,
            "audio_start_seconds": audio_edges[bin_index],
            "audio_end_seconds": audio_edges[bin_index + 1],
            "spacecraft_start_time": record["times"][np.searchsorted(source_offsets, left, side="left")],
            "spacecraft_end_time": record["times"][min(np.searchsorted(source_offsets, right, side="right"), len(record["times"])) - 1],
            "representative_vp": representative_values[bin_index],
            "valid_data_fraction": valid_fraction,
            "accepted": bool(np.isfinite(representative_values[bin_index])),
        })

    normalized_values = min_max_normalize(representative_values)
    midi_notes = normalized_to_midi_note(normalized_values, config.min_midi_note, config.max_midi_note)
    frequencies = midi_to_frequency(midi_notes)

    for row, normalized_value, midi_note, frequency in zip(rows, normalized_values, midi_notes, frequencies):
        row["normalized_vp"] = normalized_value
        row["midi_note"] = int(midi_note) if np.isfinite(midi_note) else ""
        row["note_name"] = midi_to_note_name(midi_note) if np.isfinite(midi_note) else ""
        row["frequency_hz"] = float(frequency) if np.isfinite(frequency) else np.nan

    return rows


aggregated_lookup_tables = {
    config.label: aggregate_vp_to_bins(vp_record, config)
    for config in discrete_event_configs
}

for config in discrete_event_configs:
    rows = aggregated_lookup_tables[config.label]
    accepted_count = sum(row["accepted"] for row in rows)
    print(f"{config.label}: {accepted_count}/{len(rows)} accepted bins")

In [ ]:
lookup_columns = [
    "bin_index",
    "audio_start_seconds",
    "audio_end_seconds",
    "spacecraft_start_time",
    "spacecraft_end_time",
    "representative_vp",
    "normalized_vp",
    "midi_note",
    "note_name",
    "frequency_hz",
    "valid_data_fraction",
]

print("Example lookup table: 60 notes over 60 seconds")
display_markdown_table(aggregated_lookup_tables["60 notes over 60 seconds"], columns=lookup_columns, max_rows=20)

## Discrete Event Synthesis

Every accepted bin becomes one note with constant amplitude. Rejected bins are exact silence. The short attack and release are identical for every valid note and exist only to avoid clicks at event boundaries; they do not encode `Vp` or any other scientific variable.

The tone is a simple additive waveform with a quiet second harmonic for modest warmth. It is still intentionally plain: no reverb, chords, accompaniment, musical key, or multiple instruments.

In [ ]:
def fixed_attack_release_envelope(sample_count, sample_rate, attack_seconds, release_seconds):
    """Create a fixed click-prevention envelope for one note."""
    envelope = np.ones(sample_count, dtype=float)
    attack_samples = min(int(round(attack_seconds * sample_rate)), sample_count)
    release_samples = min(int(round(release_seconds * sample_rate)), sample_count)

    if attack_samples > 0:
        envelope[:attack_samples] = np.linspace(0.0, 1.0, attack_samples, endpoint=False)
    if release_samples > 0:
        envelope[-release_samples:] = np.minimum(
            envelope[-release_samples:],
            np.linspace(1.0, 0.0, release_samples, endpoint=False),
        )

    return envelope


def synthesize_discrete_event_audio(lookup_rows, config):
    """Synthesize fixed-amplitude discrete pitch events from a lookup table."""
    total_samples = int(round(config.playback_duration_seconds * config.sample_rate))
    audio = np.zeros(total_samples, dtype=float)

    for row in lookup_rows:
        if not row["accepted"] or not np.isfinite(row["frequency_hz"]):
            continue

        start_sample = int(round(row["audio_start_seconds"] * config.sample_rate))
        end_sample = int(round(row["audio_end_seconds"] * config.sample_rate))
        start_sample = max(0, min(start_sample, total_samples))
        end_sample = max(start_sample, min(end_sample, total_samples))
        sample_count = end_sample - start_sample
        if sample_count == 0:
            continue

        t = np.arange(sample_count, dtype=float) / config.sample_rate
        fundamental = np.sin(2 * np.pi * row["frequency_hz"] * t)
        second_harmonic = 0.18 * np.sin(2 * np.pi * row["frequency_hz"] * 2.0 * t)
        tone = fundamental + second_harmonic
        tone /= np.max(np.abs(tone))
        envelope = fixed_attack_release_envelope(
            sample_count,
            config.sample_rate,
            config.attack_seconds,
            config.release_seconds,
        )
        audio[start_sample:end_sample] = config.amplitude * tone * envelope

    return audio


discrete_audio_outputs = {
    config.label: synthesize_discrete_event_audio(aggregated_lookup_tables[config.label], config)
    for config in discrete_event_configs
}

for config in discrete_event_configs:
    print(config.label)
    display(Audio(discrete_audio_outputs[config.label], rate=config.sample_rate))

## Discrete Event Comparison Plots

The plots below show the original `Vp`, the aggregated representative values, the resulting chromatic note sequence, and bins rejected because they did not meet the minimum valid-data fraction. The plotted aggregation is a control representation for listening; the original `Vp` samples remain unchanged.

In [ ]:
def plot_discrete_event_summary(config, lookup_rows):
    audio_midpoints = np.array([
        0.5 * (row["audio_start_seconds"] + row["audio_end_seconds"])
        for row in lookup_rows
    ])
    representative_vp = np.array([row["representative_vp"] for row in lookup_rows], dtype=float)
    midi_notes = np.array([
        row["midi_note"] if row["midi_note"] != "" else np.nan
        for row in lookup_rows
    ], dtype=float)
    accepted = np.array([row["accepted"] for row in lookup_rows], dtype=bool)
    valid_fraction = np.array([row["valid_data_fraction"] for row in lookup_rows], dtype=float)

    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=False)

    axes[0].plot(vp_record["times"], vp_record["nan_values"], linewidth=1, color="black")
    axes[0].set_title(f"Original Vp source samples ({config.label})")
    axes[0].set_ylabel(vp_record["units"])
    axes[0].grid(alpha=0.3)

    axes[1].plot(audio_midpoints, representative_vp, marker="o", linewidth=1)
    axes[1].set_title("Aggregated median Vp per accepted bin")
    axes[1].set_ylabel(vp_record["units"])
    axes[1].grid(alpha=0.3)

    axes[2].step(audio_midpoints, midi_notes, where="mid")
    axes[2].set_title("Resulting discrete chromatic note sequence")
    axes[2].set_ylabel("MIDI note")
    axes[2].grid(alpha=0.3)

    axes[3].bar(audio_midpoints, valid_fraction, width=config.playback_duration_seconds / config.note_count * 0.85)
    axes[3].axhline(config.minimum_valid_fraction, color="tab:red", linestyle="--", label="minimum valid fraction")
    axes[3].scatter(audio_midpoints[~accepted], valid_fraction[~accepted], color="tab:red", label="rejected/silent bin")
    axes[3].set_title("Missing or rejected bins")
    axes[3].set_xlabel("Audio time (seconds)")
    axes[3].set_ylabel("valid fraction")
    axes[3].set_ylim(0, 1.05)
    axes[3].legend()
    axes[3].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


for config in discrete_event_configs:
    plot_discrete_event_summary(config, aggregated_lookup_tables[config.label])

## Discrete Prototype Notes

This prototype changes the synthesis model rather than fine-tuning the continuous one. The central question for the next listening pass is whether temporal aggregation and semitone-spaced events make `Vp` trends easier to follow and more comfortable over 60 seconds.

This notebook still does not add `Np`, `Tpr`, MIDI export, multiple instruments, musical scales, event-detection accents, chords, accompaniment, or reverb.

## Graph and Listening Interpretation — Discrete Pitch Prototype

The following interpretation combines the latest graph review with qualitative listening feedback from the discrete pitch-event prototype. It is not a formal perceptual study.

- The discrete-note approach is much less robotic and more listenable than the original continuous-frequency prototype.
- Flat sections in the MIDI-note plot are largely caused by multiple different `Vp` bin medians being quantized to the same chromatic note.
- The current synthesizer retriggers identical consecutive notes, creating repeated attacks that falsely imply new activity.
- Sudden note jumps are caused by both real changes in aggregated `Vp` and semitone-boundary quantization.
- The current MIDI range of approximately 60-76 compresses the physical range into too few pitch levels, making significance harder to judge.
- Increasing the number of temporal bins alone will not solve the problem if pitch resolution remains coarse.
- Familiar musical intervals and progressions may distract from scientific interpretation.

These observations separate three different design layers:

- Temporal aggregation controls how many spacecraft samples contribute to each auditory state.
- Pitch quantization controls how aggregated values are rounded or not rounded into pitch levels.
- Synthesis retriggering controls whether consecutive states are heard as sustained continuity or as repeated note attacks.

The next comparison therefore tests sustained states, pitch resolution, and aggregated pitch trajectories separately rather than only increasing the number of bins.

## Revised Research Question — Sustained Aggregated Pitch

Can sustained states, greater pitch resolution, and continuous aggregated pitch targets improve interpretability without returning to the harsh sample-by-sample frequency modulation of the original prototype?

The purpose is to compare sustained chromatic states, increased temporal resolution, wider pitch span, and aggregated microtonal contours. This notebook does not automatically choose a winner. The most informative comparison is expected to be Clip C versus Clip D: Clip C asks whether more chromatic pitch levels solve the significance problem, while Clip D asks whether avoiding semitone rounding preserves detail and reduces accidental musical patterns.

## Common Normalization Reference

All clips in this comparison use one fixed `Vp` normalization reference derived from the valid `Vp` values in this file. Bin configurations are not independently min-max normalized. This is necessary so the same physical `Vp` value maps to the same auditory position in every clip.

The reference is the observed valid `Vp` range after the Notebook 03 cleaning rules are applied. Values are clipped to this reference range during auditory mapping, but the original cleaned `Vp` values remain unchanged for lookup and analysis.

In [ ]:
vp_reference_min = float(np.nanmin(vp_record["nan_values"]))
vp_reference_max = float(np.nanmax(vp_record["nan_values"]))


def normalize_with_reference(values, reference_min, reference_max):
    """Normalize values using a fixed physical reference range while preserving NaN gaps."""
    values = np.asarray(values, dtype=float)
    normalized = np.full_like(values, np.nan, dtype=float)
    finite_mask = np.isfinite(values)

    if reference_max <= reference_min:
        raise ValueError("reference_max must be greater than reference_min")

    normalized[finite_mask] = (values[finite_mask] - reference_min) / (reference_max - reference_min)
    normalized[finite_mask] = np.clip(normalized[finite_mask], 0.0, 1.0)
    return normalized


print(f"Fixed Vp normalization reference: {vp_reference_min:.2f} to {vp_reference_max:.2f} {vp_record['units']}")

## Sustained-State Comparison Configurations

The aggregation framework is preserved: 60-second playback duration, median `Vp` per bin, a minimum valid-data fraction threshold, missing or rejected bins as silence, and lookup tables linking audio events to spacecraft intervals.

For chromatic mappings, consecutive bins that map to the same MIDI note are merged and rendered as one sustained note. A new attack occurs only when pitch changes, a valid segment begins after silence, or playback begins. For changed pitch targets, short linear glides can be enabled. Glides affect only the synthesis control signal; they do not alter source data and never cross missing-data gaps.

Clip D keeps temporal aggregation but avoids semitone rounding. Adjacent microtonal targets may be treated as the same sustained state if they differ by less than the configurable cents threshold.

In [ ]:
@dataclass(frozen=True)
class SustainedPitchConfig:
    """Configuration for sustained aggregated Vp pitch comparisons."""

    label: str
    clip_id: str
    purpose: str
    note_count: int
    pitch_mode: str
    min_midi_note: int
    max_midi_note: int
    glide_seconds: float
    playback_duration_seconds: float = 60.0
    minimum_valid_fraction: float = 0.5
    representative_statistic: str = "median"
    amplitude: float = 0.18
    attack_seconds: float = 0.015
    release_seconds: float = 0.035
    sample_rate: int = 22050
    microtonal_change_threshold_cents: float = 15.0


sustained_configs = [
    SustainedPitchConfig(
        label="Clip A - 120 bins - MIDI 60-76 - merged - no glide",
        clip_id="A",
        purpose="isolate the effect of removing repeated attacks",
        note_count=120,
        pitch_mode="chromatic",
        min_midi_note=60,
        max_midi_note=76,
        glide_seconds=0.0,
    ),
    SustainedPitchConfig(
        label="Clip B - 240 bins - MIDI 60-76 - merged - short glide",
        clip_id="B",
        purpose="test increased temporal detail while retaining the current pitch range",
        note_count=240,
        pitch_mode="chromatic",
        min_midi_note=60,
        max_midi_note=76,
        glide_seconds=0.075,
    ),
    SustainedPitchConfig(
        label="Clip C - 240 bins - MIDI 48-84 - merged - short glide",
        clip_id="C",
        purpose="test whether a wider three-octave span improves perception of magnitude and significance",
        note_count=240,
        pitch_mode="chromatic",
        min_midi_note=48,
        max_midi_note=84,
        glide_seconds=0.075,
    ),
    SustainedPitchConfig(
        label="Clip D - 240 bins - microtonal targets - short glide",
        clip_id="D",
        purpose="test aggregated continuous pitch without semitone quantization",
        note_count=240,
        pitch_mode="microtonal",
        min_midi_note=48,
        max_midi_note=84,
        glide_seconds=0.075,
    ),
]

for config in sustained_configs:
    print(f"{config.clip_id}: {config.purpose}")

In [ ]:
def aggregate_vp_with_reference(record, config, reference_min, reference_max):
    """Aggregate cleaned Vp into bins using a shared normalization reference."""
    if config.note_count <= 0:
        raise ValueError("note_count must be positive")
    if not 0 <= config.minimum_valid_fraction <= 1:
        raise ValueError("minimum_valid_fraction must be between 0 and 1")
    if config.representative_statistic != "median":
        raise ValueError("Only median aggregation is implemented for this prototype")

    values = np.asarray(record["nan_values"], dtype=float)
    valid_mask = np.asarray(record["mask"], dtype=bool) & np.isfinite(values)
    source_offsets = np.asarray(record["source_offsets_seconds"], dtype=float)
    source_edges = np.linspace(float(source_offsets[0]), float(source_offsets[-1]), config.note_count + 1)
    audio_edges = np.linspace(0.0, config.playback_duration_seconds, config.note_count + 1)

    representative_values = np.full(config.note_count, np.nan, dtype=float)
    vp_min_values = np.full(config.note_count, np.nan, dtype=float)
    vp_max_values = np.full(config.note_count, np.nan, dtype=float)
    valid_fractions = np.zeros(config.note_count, dtype=float)
    rows = []

    for bin_index in range(config.note_count):
        left = source_edges[bin_index]
        right = source_edges[bin_index + 1]
        if bin_index == config.note_count - 1:
            in_bin = (source_offsets >= left) & (source_offsets <= right)
        else:
            in_bin = (source_offsets >= left) & (source_offsets < right)

        sample_indices = np.flatnonzero(in_bin)
        valid_indices = np.flatnonzero(in_bin & valid_mask)
        sample_count = int(len(sample_indices))
        valid_count = int(len(valid_indices))
        valid_fraction = valid_count / sample_count if sample_count else 0.0
        valid_fractions[bin_index] = valid_fraction

        if sample_count:
            spacecraft_start_time = record["times"][sample_indices[0]]
            spacecraft_end_time = record["times"][sample_indices[-1]]
        else:
            nearest_index = min(np.searchsorted(source_offsets, left), len(source_offsets) - 1)
            spacecraft_start_time = record["times"][nearest_index]
            spacecraft_end_time = record["times"][nearest_index]

        if valid_count and valid_fraction >= config.minimum_valid_fraction:
            valid_values = values[valid_indices]
            representative_values[bin_index] = float(np.nanmedian(valid_values))
            vp_min_values[bin_index] = float(np.nanmin(valid_values))
            vp_max_values[bin_index] = float(np.nanmax(valid_values))

        rows.append({
            "bin_index": bin_index,
            "audio_start_seconds": audio_edges[bin_index],
            "audio_end_seconds": audio_edges[bin_index + 1],
            "spacecraft_start_time": spacecraft_start_time,
            "spacecraft_end_time": spacecraft_end_time,
            "representative_vp": representative_values[bin_index],
            "vp_min": vp_min_values[bin_index],
            "vp_max": vp_max_values[bin_index],
            "valid_data_fraction": valid_fraction,
            "accepted": bool(np.isfinite(representative_values[bin_index])),
        })

    normalized_values = normalize_with_reference(representative_values, reference_min, reference_max)
    normalized_min_values = normalize_with_reference(vp_min_values, reference_min, reference_max)
    normalized_max_values = normalize_with_reference(vp_max_values, reference_min, reference_max)

    for row, normalized, normalized_min, normalized_max in zip(rows, normalized_values, normalized_min_values, normalized_max_values):
        row["normalized_vp"] = normalized
        row["normalized_min"] = normalized_min
        row["normalized_max"] = normalized_max

    return rows


def apply_pitch_mapping_to_bins(rows, config):
    """Add chromatic or microtonal pitch targets to aggregated bin rows."""
    normalized_values = np.array([row["normalized_vp"] for row in rows], dtype=float)

    if config.pitch_mode == "chromatic":
        midi_notes = normalized_to_midi_note(normalized_values, config.min_midi_note, config.max_midi_note)
        frequencies = midi_to_frequency(midi_notes)
        for row, midi_note, frequency in zip(rows, midi_notes, frequencies):
            if np.isfinite(midi_note):
                row["midi_note"] = int(midi_note)
                row["note_name"] = midi_to_note_name(midi_note)
                row["frequency_hz"] = float(frequency)
            else:
                row["midi_note"] = ""
                row["note_name"] = ""
                row["frequency_hz"] = np.nan
        return rows

    if config.pitch_mode == "microtonal":
        min_frequency = float(midi_to_frequency(config.min_midi_note))
        max_frequency = float(midi_to_frequency(config.max_midi_note))
        frequency_ratio = max_frequency / min_frequency
        frequencies = np.full_like(normalized_values, np.nan, dtype=float)
        finite_mask = np.isfinite(normalized_values)
        frequencies[finite_mask] = min_frequency * (frequency_ratio ** normalized_values[finite_mask])
        for row, frequency in zip(rows, frequencies):
            row["midi_note"] = ""
            row["note_name"] = "microtonal"
            row["frequency_hz"] = float(frequency) if np.isfinite(frequency) else np.nan
        return rows

    raise ValueError("pitch_mode must be 'chromatic' or 'microtonal'")


sustained_bin_tables = {}
for config in sustained_configs:
    rows = aggregate_vp_with_reference(vp_record, config, vp_reference_min, vp_reference_max)
    sustained_bin_tables[config.clip_id] = apply_pitch_mapping_to_bins(rows, config)

for config in sustained_configs:
    accepted_count = sum(row["accepted"] for row in sustained_bin_tables[config.clip_id])
    print(f"Clip {config.clip_id}: {accepted_count}/{config.note_count} accepted bins")

In [ ]:
def cents_between(frequency_a, frequency_b):
    """Return absolute distance between two frequencies in cents."""
    if frequency_a <= 0 or frequency_b <= 0:
        return np.inf
    return abs(1200.0 * np.log2(frequency_b / frequency_a))


def should_merge_with_event(row, event, config):
    """Decide whether a bin should extend the current audible event."""
    if not row["accepted"] or not np.isfinite(row["frequency_hz"]):
        return False

    if config.pitch_mode == "chromatic":
        return row["midi_note"] == event["midi_note"]

    previous_frequency = event["frequency_end_hz"]
    current_frequency = row["frequency_hz"]
    return cents_between(previous_frequency, current_frequency) < config.microtonal_change_threshold_cents


def merge_bins_to_audible_events(rows, config):
    """Merge consecutive compatible bins into sustained audible events."""
    events = []
    current_event = None

    for row in rows:
        if not row["accepted"] or not np.isfinite(row["frequency_hz"]):
            if current_event is not None:
                events.append(current_event)
                current_event = None
            continue

        if current_event is not None and should_merge_with_event(row, current_event, config):
            current_event["audio_end_seconds"] = row["audio_end_seconds"]
            current_event["spacecraft_end_time"] = row["spacecraft_end_time"]
            current_event["source_bin_count"] += 1
            current_event["representative_vp_values"].append(row["representative_vp"])
            current_event["vp_min"] = min(current_event["vp_min"], row["vp_min"])
            current_event["vp_max"] = max(current_event["vp_max"], row["vp_max"])
            current_event["normalized_values"].append(row["normalized_vp"])
            current_event["frequency_values"].append(row["frequency_hz"])
            current_event["frequency_end_hz"] = row["frequency_hz"]
            current_event["valid_fraction_values"].append(row["valid_data_fraction"])
            current_event["source_bin_indices"].append(row["bin_index"])
        else:
            if current_event is not None:
                events.append(current_event)
            current_event = {
                "audio_start_seconds": row["audio_start_seconds"],
                "audio_end_seconds": row["audio_end_seconds"],
                "spacecraft_start_time": row["spacecraft_start_time"],
                "spacecraft_end_time": row["spacecraft_end_time"],
                "source_bin_count": 1,
                "source_bin_indices": [row["bin_index"]],
                "representative_vp_values": [row["representative_vp"]],
                "vp_min": row["vp_min"],
                "vp_max": row["vp_max"],
                "normalized_values": [row["normalized_vp"]],
                "midi_note": row["midi_note"],
                "note_name": row["note_name"],
                "frequency_values": [row["frequency_hz"]],
                "frequency_start_hz": row["frequency_hz"],
                "frequency_end_hz": row["frequency_hz"],
                "valid_fraction_values": [row["valid_data_fraction"]],
                "glide_used": bool(config.glide_seconds > 0),
            }

    if current_event is not None:
        events.append(current_event)

    lookup_events = []
    for event_index, event in enumerate(events):
        representative_values = np.array(event["representative_vp_values"], dtype=float)
        normalized_values = np.array(event["normalized_values"], dtype=float)
        frequency_values = np.array(event["frequency_values"], dtype=float)
        valid_fraction_values = np.array(event["valid_fraction_values"], dtype=float)
        lookup_events.append({
            "event_index": event_index,
            "audio_start_seconds": event["audio_start_seconds"],
            "audio_end_seconds": event["audio_end_seconds"],
            "spacecraft_start_time": event["spacecraft_start_time"],
            "spacecraft_end_time": event["spacecraft_end_time"],
            "source_bin_count": event["source_bin_count"],
            "representative_vp_median": float(np.nanmedian(representative_values)),
            "vp_min": event["vp_min"],
            "vp_max": event["vp_max"],
            "normalized_value_or_range": (
                f"{np.nanmin(normalized_values):.3f}-{np.nanmax(normalized_values):.3f}"
                if np.nanmin(normalized_values) != np.nanmax(normalized_values)
                else f"{normalized_values[0]:.3f}"
            ),
            "midi_note": event["midi_note"],
            "note_name": event["note_name"],
            "frequency_or_range_hz": (
                f"{np.nanmin(frequency_values):.2f}-{np.nanmax(frequency_values):.2f}"
                if np.nanmin(frequency_values) != np.nanmax(frequency_values)
                else f"{frequency_values[0]:.2f}"
            ),
            "frequency_start_hz": event["frequency_start_hz"],
            "frequency_end_hz": event["frequency_end_hz"],
            "valid_data_fraction": float(np.nanmean(valid_fraction_values)),
            "glide_used": event["glide_used"],
            "source_bin_indices": event["source_bin_indices"],
            "frequency_values": event["frequency_values"],
        })

    return lookup_events


sustained_event_tables = {
    config.clip_id: merge_bins_to_audible_events(sustained_bin_tables[config.clip_id], config)
    for config in sustained_configs
}

for config in sustained_configs:
    source_bins = len(sustained_bin_tables[config.clip_id])
    audible_events = len(sustained_event_tables[config.clip_id])
    print(f"Clip {config.clip_id}: {source_bins} source bins -> {audible_events} audible events")
    if config.pitch_mode == "microtonal":
        print(f"  Microtonal merge threshold: {config.microtonal_change_threshold_cents:.1f} cents")

## Sustained-State Synthesis

All valid sound uses constant, data-independent amplitude. The envelope is rendering behavior, not a scientific loudness mapping. It is applied only at audible event starts and ends to prevent clicks; identical consecutive chromatic notes are sustained without repeated envelope attacks.

Short glides, when enabled, interpolate between adjacent pitch targets within a valid audible event or into a changed target. They never cross missing-data gaps and do not alter the aggregated `Vp` values.

In [ ]:
def event_pitch_trajectory(event, config, sample_count):
    """Create the pitch-control trajectory for one merged event."""
    if sample_count <= 0:
        return np.array([], dtype=float)

    frequency_values = np.asarray(event["frequency_values"], dtype=float)
    if len(frequency_values) == 1:
        return np.full(sample_count, frequency_values[0], dtype=float)

    bin_duration_seconds = config.playback_duration_seconds / config.note_count
    event_duration_seconds = event["audio_end_seconds"] - event["audio_start_seconds"]
    local_times = np.arange(sample_count, dtype=float) / config.sample_rate
    target_times = np.arange(len(frequency_values), dtype=float) * bin_duration_seconds
    target_times = np.clip(target_times, 0.0, max(event_duration_seconds - 1.0 / config.sample_rate, 0.0))

    if config.glide_seconds <= 0:
        positions = np.minimum((local_times / bin_duration_seconds).astype(int), len(frequency_values) - 1)
        return frequency_values[positions]

    trajectory = np.full(sample_count, frequency_values[0], dtype=float)
    for target_index in range(1, len(frequency_values)):
        previous_frequency = frequency_values[target_index - 1]
        next_frequency = frequency_values[target_index]
        change_time = target_times[target_index]
        glide_start = change_time
        glide_end = min(change_time + config.glide_seconds, event_duration_seconds)

        before_next = (local_times >= change_time) & (local_times < glide_start)
        trajectory[before_next] = previous_frequency

        in_glide = (local_times >= glide_start) & (local_times < glide_end)
        if np.any(in_glide) and glide_end > glide_start:
            glide_fraction = (local_times[in_glide] - glide_start) / (glide_end - glide_start)
            trajectory[in_glide] = previous_frequency + glide_fraction * (next_frequency - previous_frequency)

        after_glide = local_times >= glide_end
        trajectory[after_glide] = next_frequency

    return trajectory


def synthesize_sustained_events(event_rows, config):
    """Synthesize merged sustained pitch events with optional short glides."""
    total_samples = int(round(config.playback_duration_seconds * config.sample_rate))
    audio = np.zeros(total_samples, dtype=float)
    pitch_control = np.full(total_samples, np.nan, dtype=float)
    boundary_samples = []

    for event in event_rows:
        start_sample = int(round(event["audio_start_seconds"] * config.sample_rate))
        end_sample = int(round(event["audio_end_seconds"] * config.sample_rate))
        start_sample = max(0, min(start_sample, total_samples))
        end_sample = max(start_sample, min(end_sample, total_samples))
        sample_count = end_sample - start_sample
        if sample_count <= 0:
            continue

        frequency_trajectory = event_pitch_trajectory(event, config, sample_count)
        t = np.arange(sample_count, dtype=float) / config.sample_rate
        phase = 2 * np.pi * np.cumsum(frequency_trajectory) / config.sample_rate
        fundamental = np.sin(phase)
        second_harmonic = 0.18 * np.sin(2.0 * phase)
        tone = fundamental + second_harmonic
        tone /= np.max(np.abs(tone))
        envelope = fixed_attack_release_envelope(sample_count, config.sample_rate, config.attack_seconds, config.release_seconds)

        audio[start_sample:end_sample] = config.amplitude * tone * envelope
        pitch_control[start_sample:end_sample] = frequency_trajectory
        boundary_samples.append(start_sample)

    return audio, pitch_control, np.array(boundary_samples, dtype=int)


sustained_audio_outputs = {}
sustained_pitch_controls = {}
sustained_boundary_samples = {}

for config in sustained_configs:
    audio, pitch_control, boundary_samples = synthesize_sustained_events(sustained_event_tables[config.clip_id], config)
    sustained_audio_outputs[config.clip_id] = audio
    sustained_pitch_controls[config.clip_id] = pitch_control
    sustained_boundary_samples[config.clip_id] = boundary_samples

for config in sustained_configs:
    print(config.label)
    display(Audio(sustained_audio_outputs[config.clip_id], rate=config.sample_rate))

## Sustained-State Lookup Tables

Each audible event keeps a lookup back to the source spacecraft interval and the bins that contributed to the event. Chromatic clips include MIDI note and note name. The microtonal clip leaves MIDI note blank and reports frequency ranges instead.

In [ ]:
sustained_lookup_columns = [
    "event_index",
    "audio_start_seconds",
    "audio_end_seconds",
    "spacecraft_start_time",
    "spacecraft_end_time",
    "source_bin_count",
    "representative_vp_median",
    "vp_min",
    "vp_max",
    "normalized_value_or_range",
    "midi_note",
    "note_name",
    "frequency_or_range_hz",
    "valid_data_fraction",
    "glide_used",
]

for config in sustained_configs:
    print(config.label)
    display_markdown_table(sustained_event_tables[config.clip_id], columns=sustained_lookup_columns, max_rows=18)

## Sustained-State Comparison Plots

For each configuration, the plots show aggregated median `Vp`, normalized targets, mapped pitch targets, final pitch-control trajectory, merged event boundaries, and missing or rejected bins. Chromatic clips show the MIDI staircase; the microtonal clip shows the unrounded frequency trajectory.

In [ ]:
def plot_sustained_config(config, bin_rows, event_rows, pitch_control):
    audio_midpoints = np.array([
        0.5 * (row["audio_start_seconds"] + row["audio_end_seconds"])
        for row in bin_rows
    ])
    representative_vp = np.array([row["representative_vp"] for row in bin_rows], dtype=float)
    normalized = np.array([row["normalized_vp"] for row in bin_rows], dtype=float)
    frequencies = np.array([row["frequency_hz"] for row in bin_rows], dtype=float)
    accepted = np.array([row["accepted"] for row in bin_rows], dtype=bool)
    valid_fraction = np.array([row["valid_data_fraction"] for row in bin_rows], dtype=float)
    event_boundaries = np.array([row["audio_start_seconds"] for row in event_rows], dtype=float)

    fig, axes = plt.subplots(6, 1, figsize=(12, 13), sharex=False)

    axes[0].plot(audio_midpoints, representative_vp, marker="o", markersize=2, linewidth=1)
    axes[0].set_title(f"{config.label}: aggregated median Vp")
    axes[0].set_ylabel(vp_record["units"])
    axes[0].grid(alpha=0.3)

    axes[1].plot(audio_midpoints, normalized, marker="o", markersize=2, linewidth=1)
    axes[1].set_title("Normalized target value using common Vp reference")
    axes[1].set_ylabel("0-1")
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].grid(alpha=0.3)

    if config.pitch_mode == "chromatic":
        midi_values = np.array([row["midi_note"] if row["midi_note"] != "" else np.nan for row in bin_rows], dtype=float)
        axes[2].step(audio_midpoints, midi_values, where="mid")
        axes[2].set_title("Mapped pitch target: MIDI staircase")
        axes[2].set_ylabel("MIDI note")
    else:
        axes[2].plot(audio_midpoints, frequencies, marker="o", markersize=2, linewidth=1)
        axes[2].set_title("Mapped pitch target: unrounded microtonal frequency")
        axes[2].set_ylabel("Hz")
    axes[2].grid(alpha=0.3)

    pitch_time = np.arange(len(pitch_control), dtype=float) / config.sample_rate
    axes[3].plot(pitch_time, pitch_control, linewidth=1)
    axes[3].set_title("Final pitch-control trajectory used by synthesizer")
    axes[3].set_ylabel("Hz")
    axes[3].grid(alpha=0.3)

    for boundary in event_boundaries:
        axes[4].axvline(boundary, color="tab:purple", alpha=0.25, linewidth=0.8)
    axes[4].set_title(f"Merged event boundaries ({len(event_rows)} audible events)")
    axes[4].set_xlim(0, config.playback_duration_seconds)
    axes[4].set_ylabel("boundary")
    axes[4].set_yticks([])
    axes[4].grid(alpha=0.3)

    axes[5].bar(audio_midpoints, valid_fraction, width=config.playback_duration_seconds / config.note_count * 0.85)
    axes[5].axhline(config.minimum_valid_fraction, color="tab:red", linestyle="--", label="minimum valid fraction")
    axes[5].scatter(audio_midpoints[~accepted], valid_fraction[~accepted], color="tab:red", label="rejected/silent bin")
    axes[5].set_title("Missing or rejected bins")
    axes[5].set_xlabel("Audio time (seconds)")
    axes[5].set_ylabel("valid fraction")
    axes[5].set_ylim(0, 1.05)
    axes[5].legend()
    axes[5].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


for config in sustained_configs:
    plot_sustained_config(
        config,
        sustained_bin_tables[config.clip_id],
        sustained_event_tables[config.clip_id],
        sustained_pitch_controls[config.clip_id],
    )

## Current Comparison Focus

This comparison keeps amplitude independent of the data and treats the envelope as fixed rendering behavior, not scientific loudness mapping. Missing or rejected bins remain silent. The experiment is limited to `Vp` and does not add `Np`, `Tpr`, MIDI export, instruments, reverb, chords, musical keys, event-detection accents, or data-driven loudness.

The next listening pass should focus especially on Clip C versus Clip D. Clip C tests whether more chromatic pitch levels solve the significance problem. Clip D tests whether avoiding semitone rounding preserves detail and reduces accidental musical patterns while retaining temporal aggregation.

## Human Listening Feedback — Wide-Range and Microtonal Comparison

The following notes are qualitative listening observations from the wide-range chromatic and aggregated microtonal comparison. They are not a formal perceptual study.

- The sustained-state behavior of the discrete and aggregated clips is a major improvement because identical consecutive values remain continuous rather than being retriggered.
- This continuity substantially reduces popping and false rhythmic activity.
- Both the wide chromatic clip and the continuous microtonal clip still sound too musical.
- The wider pitch span makes familiar intervals and melodic patterns more noticeable, which distracts from the scientific data.
- In the MIDI 48-84 clip, the first four audible notes were perceived as the first four notes of a minor scale.
- The listener has musical training and automatically attends to recognizable scales, intervals, and progressions.
- This suggests that increasing pitch range may improve numerical contrast while simultaneously reducing scientific focus.
- A narrower pitch range may keep nearby values perceptually close while allowing genuinely larger changes to remain relatively more noticeable.
- The preferred future direction is likely a hybrid of the original sine-wave contour and the sustained-state behavior of the discrete prototypes.

## Revised Research Question — Narrow Sustained Pitch

Can a compressed pitch range preserve relative `Vp` changes while reducing accidental melodic interpretation and retaining the continuity benefits of sustained pitch states?

This comparison keeps the same data preparation pipeline for every clip: 60-second playback, the same `Vp` source data, the same fixed normalization reference, 240 aggregation bins, the same valid-fraction threshold, missing or rejected bins as silence, constant amplitude, and identical fixed attack/release behavior only for click prevention. Individual clips are not independently normalized.

Only three design factors vary: continuous versus chromatic pitch targets, timbre, and pitch span.

## Narrow-Range Comparison Configuration

Every new prototype preserves sustained states. Consecutive pitch targets that are identical or effectively unchanged are not retriggered. The existing tone is sustained through those intervals. A new envelope begins only when playback begins, valid data resumes after silence, or the quantized chromatic target changes.

For sustained continuous targets, adjacent pitch targets within a small cents threshold are treated as the same state. Changes below this threshold are neither glided nor retriggered. Changes above the threshold use a short linear glide in the synthesis control signal only. Glides never cross missing-data gaps and do not alter the source data.

In [ ]:
@dataclass(frozen=True)
class NarrowPitchConfig:
    """Configuration for narrow sustained aggregated Vp pitch comparisons."""

    label: str
    clip_id: str
    purpose: str
    pitch_mode: str
    timbre: str
    note_count: int = 240
    playback_duration_seconds: float = 60.0
    minimum_valid_fraction: float = 0.5
    representative_statistic: str = "median"
    amplitude: float = 0.18
    attack_seconds: float = 0.015
    release_seconds: float = 0.035
    sample_rate: int = 22050
    glide_seconds: float = 0.075
    continuous_min_frequency_hz: float = 330.0
    continuous_max_frequency_hz: float = 440.0
    min_midi_note: int = 64
    max_midi_note: int = 70
    continuous_change_threshold_cents: float = 15.0


narrow_configs = [
    NarrowPitchConfig(
        label="Clip E - Aggregated continuous sine pitch - 240 bins - sustained states - short glide - 330-440 Hz",
        clip_id="E",
        purpose="test a restrained continuous contour while preserving the simple sine-wave reference",
        pitch_mode="continuous",
        timbre="sine",
        continuous_min_frequency_hz=330.0,
        continuous_max_frequency_hz=440.0,
    ),
    NarrowPitchConfig(
        label="Clip F - Aggregated continuous harmonic pitch - 240 bins - sustained states - short glide - 330-440 Hz",
        clip_id="F",
        purpose="determine whether timbre or pitch span is responsible for discomfort",
        pitch_mode="continuous",
        timbre="harmonic",
        continuous_min_frequency_hz=330.0,
        continuous_max_frequency_hz=440.0,
    ),
    NarrowPitchConfig(
        label="Clip G - Quantized chromatic pitch - 240 bins - sustained repeated notes - MIDI 64-70",
        clip_id="G",
        purpose="test whether coarse but restrained chromatic mapping reduces melodic distraction",
        pitch_mode="chromatic",
        timbre="harmonic",
        min_midi_note=64,
        max_midi_note=70,
        glide_seconds=0.0,
    ),
    NarrowPitchConfig(
        label="Clip H - Aggregated continuous pitch - 240 bins - sustained states - short glide - 300-500 Hz",
        clip_id="H",
        purpose="test the tradeoff between pitch contrast and musical distraction",
        pitch_mode="continuous",
        timbre="harmonic",
        continuous_min_frequency_hz=300.0,
        continuous_max_frequency_hz=500.0,
    ),
]

for config in narrow_configs:
    print(f"Clip {config.clip_id}: {config.purpose}")

In [ ]:
def map_normalized_to_log_frequency(normalized_values, min_frequency_hz, max_frequency_hz):
    """Map normalized values to logarithmically spaced frequency targets."""
    values = np.asarray(normalized_values, dtype=float)
    frequencies = np.full_like(values, np.nan, dtype=float)
    finite_mask = np.isfinite(values)

    if min_frequency_hz <= 0 or max_frequency_hz <= min_frequency_hz:
        raise ValueError("Frequency endpoints must be positive and increasing")

    ratio = max_frequency_hz / min_frequency_hz
    frequencies[finite_mask] = min_frequency_hz * (ratio ** values[finite_mask])
    return frequencies


def prepare_narrow_bin_rows(config):
    """Aggregate Vp once and add narrow comparison pitch targets."""
    rows = aggregate_vp_with_reference(vp_record, config, vp_reference_min, vp_reference_max)
    normalized = np.array([row["normalized_vp"] for row in rows], dtype=float)

    if config.pitch_mode == "continuous":
        frequencies = map_normalized_to_log_frequency(
            normalized,
            config.continuous_min_frequency_hz,
            config.continuous_max_frequency_hz,
        )
        for row, frequency in zip(rows, frequencies):
            row["midi_note"] = ""
            row["note_name"] = "continuous"
            row["target_frequency_hz"] = float(frequency) if np.isfinite(frequency) else np.nan
            row["state_frequency_hz"] = row["target_frequency_hz"]
            row["merged_by_threshold"] = False
        return rows

    if config.pitch_mode == "chromatic":
        midi_notes = normalized_to_midi_note(normalized, config.min_midi_note, config.max_midi_note)
        frequencies = midi_to_frequency(midi_notes)
        for row, midi_note, frequency in zip(rows, midi_notes, frequencies):
            if np.isfinite(midi_note):
                row["midi_note"] = int(midi_note)
                row["note_name"] = midi_to_note_name(midi_note)
                row["target_frequency_hz"] = float(frequency)
                row["state_frequency_hz"] = float(frequency)
            else:
                row["midi_note"] = ""
                row["note_name"] = ""
                row["target_frequency_hz"] = np.nan
                row["state_frequency_hz"] = np.nan
            row["merged_by_threshold"] = False
        return rows

    raise ValueError("pitch_mode must be 'continuous' or 'chromatic'")


narrow_bin_tables = {config.clip_id: prepare_narrow_bin_rows(config) for config in narrow_configs}

for config in narrow_configs:
    accepted_count = sum(row["accepted"] for row in narrow_bin_tables[config.clip_id])
    print(f"Clip {config.clip_id}: {accepted_count}/{config.note_count} accepted pitch targets")

In [ ]:
def apply_continuous_change_threshold(rows, config):
    """Hold adjacent continuous targets that differ by less than the cents threshold."""
    if config.pitch_mode != "continuous":
        return rows, 0

    previous_state_frequency = np.nan
    previous_was_valid = False
    suppressed_count = 0

    for row in rows:
        if not row["accepted"] or not np.isfinite(row["target_frequency_hz"]):
            previous_state_frequency = np.nan
            previous_was_valid = False
            continue

        if previous_was_valid:
            cents_change = cents_between(previous_state_frequency, row["target_frequency_hz"])
            if cents_change < config.continuous_change_threshold_cents:
                row["state_frequency_hz"] = previous_state_frequency
                row["merged_by_threshold"] = True
                suppressed_count += 1
            else:
                row["state_frequency_hz"] = row["target_frequency_hz"]
                row["merged_by_threshold"] = False
                previous_state_frequency = row["state_frequency_hz"]
        else:
            row["state_frequency_hz"] = row["target_frequency_hz"]
            row["merged_by_threshold"] = False
            previous_state_frequency = row["state_frequency_hz"]
            previous_was_valid = True

    return rows, suppressed_count


narrow_suppressed_change_counts = {}
for config in narrow_configs:
    rows, suppressed_count = apply_continuous_change_threshold(narrow_bin_tables[config.clip_id], config)
    narrow_bin_tables[config.clip_id] = rows
    narrow_suppressed_change_counts[config.clip_id] = suppressed_count
    if config.pitch_mode == "continuous":
        print(
            f"Clip {config.clip_id}: {suppressed_count} adjacent target changes suppressed "
            f"below {config.continuous_change_threshold_cents:.1f} cents"
        )

In [ ]:
def merge_narrow_rows_to_events(rows, config):
    """Create sustained events for narrow comparison synthesis and lookup."""
    events = []
    current = None

    for row in rows:
        if not row["accepted"] or not np.isfinite(row["state_frequency_hz"]):
            if current is not None:
                events.append(current)
                current = None
            continue

        can_merge = False
        if current is not None:
            if config.pitch_mode == "continuous":
                can_merge = True
            elif config.pitch_mode == "chromatic":
                can_merge = row["midi_note"] == current["midi_note"]

        if can_merge:
            current["audio_end_seconds"] = row["audio_end_seconds"]
            current["spacecraft_end_time"] = row["spacecraft_end_time"]
            current["source_bin_count"] += 1
            current["source_bin_indices"].append(row["bin_index"])
            current["representative_vp_values"].append(row["representative_vp"])
            current["vp_min"] = min(current["vp_min"], row["vp_min"])
            current["vp_max"] = max(current["vp_max"], row["vp_max"])
            current["normalized_values"].append(row["normalized_vp"])
            current["target_frequency_values"].append(row["target_frequency_hz"])
            current["state_frequency_values"].append(row["state_frequency_hz"])
            current["valid_fraction_values"].append(row["valid_data_fraction"])
            current["threshold_merge_flags"].append(row["merged_by_threshold"])
        else:
            if current is not None:
                events.append(current)
            current = {
                "audio_start_seconds": row["audio_start_seconds"],
                "audio_end_seconds": row["audio_end_seconds"],
                "spacecraft_start_time": row["spacecraft_start_time"],
                "spacecraft_end_time": row["spacecraft_end_time"],
                "source_bin_count": 1,
                "source_bin_indices": [row["bin_index"]],
                "representative_vp_values": [row["representative_vp"]],
                "vp_min": row["vp_min"],
                "vp_max": row["vp_max"],
                "normalized_values": [row["normalized_vp"]],
                "target_frequency_values": [row["target_frequency_hz"]],
                "state_frequency_values": [row["state_frequency_hz"]],
                "midi_note": row["midi_note"],
                "note_name": row["note_name"],
                "valid_fraction_values": [row["valid_data_fraction"]],
                "threshold_merge_flags": [row["merged_by_threshold"]],
            }

    if current is not None:
        events.append(current)

    lookup_events = []
    for event_index, event in enumerate(events):
        representative_values = np.asarray(event["representative_vp_values"], dtype=float)
        normalized_values = np.asarray(event["normalized_values"], dtype=float)
        target_frequencies = np.asarray(event["target_frequency_values"], dtype=float)
        state_frequencies = np.asarray(event["state_frequency_values"], dtype=float)
        valid_fraction_values = np.asarray(event["valid_fraction_values"], dtype=float)
        threshold_merge_count = int(np.sum(event["threshold_merge_flags"]))
        lookup_events.append({
            "event_index": event_index,
            "audio_start_seconds": event["audio_start_seconds"],
            "audio_end_seconds": event["audio_end_seconds"],
            "spacecraft_start_time": event["spacecraft_start_time"],
            "spacecraft_end_time": event["spacecraft_end_time"],
            "source_bin_count": event["source_bin_count"],
            "representative_vp_median": float(np.nanmedian(representative_values)),
            "vp_min": event["vp_min"],
            "vp_max": event["vp_max"],
            "normalized_range": f"{np.nanmin(normalized_values):.3f}-{np.nanmax(normalized_values):.3f}",
            "starting_frequency_hz": float(state_frequencies[0]),
            "ending_frequency_hz": float(state_frequencies[-1]),
            "frequency_range_hz": f"{np.nanmin(state_frequencies):.2f}-{np.nanmax(state_frequencies):.2f}",
            "midi_note": event["midi_note"],
            "note_name": event["note_name"],
            "valid_data_fraction": float(np.nanmean(valid_fraction_values)),
            "glide_occurred": bool(config.glide_seconds > 0 and np.nanmax(state_frequencies) != np.nanmin(state_frequencies)),
            "merged_by_cents_threshold": bool(threshold_merge_count > 0),
            "threshold_merged_target_count": threshold_merge_count,
            "source_bin_indices": event["source_bin_indices"],
            "state_frequency_values": event["state_frequency_values"],
            "target_frequency_values": event["target_frequency_values"],
        })

    return lookup_events


narrow_event_tables = {
    config.clip_id: merge_narrow_rows_to_events(narrow_bin_tables[config.clip_id], config)
    for config in narrow_configs
}

for config in narrow_configs:
    target_count = sum(row["accepted"] for row in narrow_bin_tables[config.clip_id])
    audible_attacks = len(narrow_event_tables[config.clip_id])
    print(f"Clip {config.clip_id}: {target_count} accepted pitch targets -> {audible_attacks} audible attacks")
    if config.pitch_mode == "continuous":
        print(f"  suppressed negligible changes: {narrow_suppressed_change_counts[config.clip_id]}")

## Narrow-Range Synthesis

All valid sound uses the same constant, data-independent amplitude. The envelope is rendering behavior, not a scientific loudness mapping. It is used only to prevent clicks at audible attack and release points.

Clips E, F, and H use aggregated continuous pitch targets with short glides for changes above the cents threshold. Clip E uses a pure sine tone to preserve the simplest reference. Clips F and H use the same modest-harmonic tone as the discrete prototype. Clip G uses chromatic quantization over MIDI 64-70 and sustains repeated notes without retriggering them.

In [ ]:
def narrow_event_pitch_trajectory(event, config, sample_count):
    """Create pitch-control trajectory for one narrow sustained event."""
    if sample_count <= 0:
        return np.array([], dtype=float)

    state_frequencies = np.asarray(event["state_frequency_values"], dtype=float)
    if len(state_frequencies) == 1:
        return np.full(sample_count, state_frequencies[0], dtype=float)

    bin_duration_seconds = config.playback_duration_seconds / config.note_count
    event_duration_seconds = event["audio_end_seconds"] - event["audio_start_seconds"]
    local_times = np.arange(sample_count, dtype=float) / config.sample_rate
    positions = np.minimum((local_times / bin_duration_seconds).astype(int), len(state_frequencies) - 1)
    trajectory = state_frequencies[positions]

    if config.pitch_mode != "continuous" or config.glide_seconds <= 0:
        return trajectory

    for target_index in range(1, len(state_frequencies)):
        previous_frequency = state_frequencies[target_index - 1]
        next_frequency = state_frequencies[target_index]
        if previous_frequency == next_frequency:
            continue

        change_time = target_index * bin_duration_seconds
        glide_start = change_time
        glide_end = min(change_time + config.glide_seconds, event_duration_seconds)
        in_glide = (local_times >= glide_start) & (local_times < glide_end)
        if np.any(in_glide) and glide_end > glide_start:
            glide_fraction = (local_times[in_glide] - glide_start) / (glide_end - glide_start)
            trajectory[in_glide] = previous_frequency + glide_fraction * (next_frequency - previous_frequency)

    return trajectory


def synthesize_narrow_events(event_rows, config):
    """Synthesize narrow comparison audio and final pitch-control trajectory."""
    total_samples = int(round(config.playback_duration_seconds * config.sample_rate))
    audio = np.zeros(total_samples, dtype=float)
    pitch_control = np.full(total_samples, np.nan, dtype=float)
    attack_samples = []

    for event in event_rows:
        start_sample = int(round(event["audio_start_seconds"] * config.sample_rate))
        end_sample = int(round(event["audio_end_seconds"] * config.sample_rate))
        start_sample = max(0, min(start_sample, total_samples))
        end_sample = max(start_sample, min(end_sample, total_samples))
        sample_count = end_sample - start_sample
        if sample_count <= 0:
            continue

        frequency_trajectory = narrow_event_pitch_trajectory(event, config, sample_count)
        phase = 2 * np.pi * np.cumsum(frequency_trajectory) / config.sample_rate
        if config.timbre == "sine":
            tone = np.sin(phase)
        elif config.timbre == "harmonic":
            tone = np.sin(phase) + 0.18 * np.sin(2.0 * phase)
            tone /= np.max(np.abs(tone))
        else:
            raise ValueError("Unsupported timbre")

        envelope = fixed_attack_release_envelope(sample_count, config.sample_rate, config.attack_seconds, config.release_seconds)
        audio[start_sample:end_sample] = config.amplitude * tone * envelope
        pitch_control[start_sample:end_sample] = frequency_trajectory
        attack_samples.append(start_sample)

    return audio, pitch_control, np.asarray(attack_samples, dtype=int)


narrow_audio_outputs = {}
narrow_pitch_controls = {}
narrow_attack_samples = {}

for config in narrow_configs:
    audio, pitch_control, attack_samples = synthesize_narrow_events(narrow_event_tables[config.clip_id], config)
    narrow_audio_outputs[config.clip_id] = audio
    narrow_pitch_controls[config.clip_id] = pitch_control
    narrow_attack_samples[config.clip_id] = attack_samples

for config in narrow_configs:
    print(config.label)
    display(Audio(narrow_audio_outputs[config.clip_id], rate=config.sample_rate))

## Narrow-Range Lookup Tables

Each sustained event keeps traceability to its source spacecraft interval and source bins. Continuous clips report starting, ending, and ranged frequency values. The chromatic clip also reports MIDI note and note name. The table records whether glides occurred and whether adjacent targets were merged by the cents threshold.

In [ ]:
narrow_lookup_columns = [
    "event_index",
    "audio_start_seconds",
    "audio_end_seconds",
    "spacecraft_start_time",
    "spacecraft_end_time",
    "source_bin_count",
    "representative_vp_median",
    "vp_min",
    "vp_max",
    "normalized_range",
    "starting_frequency_hz",
    "ending_frequency_hz",
    "frequency_range_hz",
    "midi_note",
    "note_name",
    "valid_data_fraction",
    "glide_occurred",
    "merged_by_cents_threshold",
    "threshold_merged_target_count",
]

for config in narrow_configs:
    print(config.label)
    display_markdown_table(narrow_event_tables[config.clip_id], columns=narrow_lookup_columns, max_rows=18)

## Narrow-Range Comparison Plots

For every clip, the plots show aggregated `Vp`, normalized target values, mapped pitch targets, final sustained or gliding pitch trajectory, audible attack positions, and missing or rejected bins. The normalized axes are identical across clips. The final pitch-control plots use a shared frequency axis to make pitch-span differences easier to compare.

In [ ]:
all_narrow_frequencies = []
for config in narrow_configs:
    for row in narrow_bin_tables[config.clip_id]:
        if np.isfinite(row["target_frequency_hz"]):
            all_narrow_frequencies.append(row["target_frequency_hz"])
common_frequency_min = min(all_narrow_frequencies) - 10
common_frequency_max = max(all_narrow_frequencies) + 10


def plot_narrow_config(config, bin_rows, event_rows, pitch_control, attack_samples):
    audio_midpoints = np.array([
        0.5 * (row["audio_start_seconds"] + row["audio_end_seconds"])
        for row in bin_rows
    ])
    representative_vp = np.array([row["representative_vp"] for row in bin_rows], dtype=float)
    normalized = np.array([row["normalized_vp"] for row in bin_rows], dtype=float)
    target_frequencies = np.array([row["target_frequency_hz"] for row in bin_rows], dtype=float)
    accepted = np.array([row["accepted"] for row in bin_rows], dtype=bool)
    valid_fraction = np.array([row["valid_data_fraction"] for row in bin_rows], dtype=float)
    attack_times = attack_samples / config.sample_rate

    fig, axes = plt.subplots(6, 1, figsize=(12, 13), sharex=False)

    axes[0].plot(audio_midpoints, representative_vp, marker="o", markersize=2, linewidth=1)
    axes[0].set_title(f"{config.label}: aggregated median Vp")
    axes[0].set_ylabel(vp_record["units"])
    axes[0].grid(alpha=0.3)

    axes[1].plot(audio_midpoints, normalized, marker="o", markersize=2, linewidth=1)
    axes[1].set_title("Normalized targets using the common Vp reference")
    axes[1].set_ylabel("0-1")
    axes[1].set_ylim(-0.05, 1.05)
    axes[1].grid(alpha=0.3)

    if config.pitch_mode == "chromatic":
        midi_values = np.array([row["midi_note"] if row["midi_note"] != "" else np.nan for row in bin_rows], dtype=float)
        axes[2].step(audio_midpoints, midi_values, where="mid")
        axes[2].set_title("Mapped pitch targets: MIDI staircase")
        axes[2].set_ylabel("MIDI note")
    else:
        axes[2].plot(audio_midpoints, target_frequencies, marker="o", markersize=2, linewidth=1)
        axes[2].set_title("Mapped pitch targets: continuous frequency")
        axes[2].set_ylabel("Hz")
        axes[2].set_ylim(common_frequency_min, common_frequency_max)
    axes[2].grid(alpha=0.3)

    pitch_time = np.arange(len(pitch_control), dtype=float) / config.sample_rate
    axes[3].plot(pitch_time, pitch_control, linewidth=1)
    axes[3].set_title("Final sustained/gliding pitch-control trajectory")
    axes[3].set_ylabel("Hz")
    axes[3].set_ylim(common_frequency_min, common_frequency_max)
    axes[3].grid(alpha=0.3)

    for attack_time in attack_times:
        axes[4].axvline(attack_time, color="tab:purple", alpha=0.35, linewidth=0.8)
    axes[4].set_title(f"Audible attack positions ({len(attack_times)} attacks)")
    axes[4].set_xlim(0, config.playback_duration_seconds)
    axes[4].set_ylabel("attack")
    axes[4].set_yticks([])
    axes[4].grid(alpha=0.3)

    axes[5].bar(audio_midpoints, valid_fraction, width=config.playback_duration_seconds / config.note_count * 0.85)
    axes[5].axhline(config.minimum_valid_fraction, color="tab:red", linestyle="--", label="minimum valid fraction")
    axes[5].scatter(audio_midpoints[~accepted], valid_fraction[~accepted], color="tab:red", label="rejected/silent bin")
    axes[5].set_title("Missing or rejected bins")
    axes[5].set_xlabel("Audio time (seconds)")
    axes[5].set_ylabel("valid fraction")
    axes[5].set_ylim(0, 1.05)
    axes[5].legend()
    axes[5].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


for config in narrow_configs:
    plot_narrow_config(
        config,
        narrow_bin_tables[config.clip_id],
        narrow_event_tables[config.clip_id],
        narrow_pitch_controls[config.clip_id],
        narrow_attack_samples[config.clip_id],
    )

## Side Experiment: Clip G MIDI Export for DAW Instrument Testing

As a small side experiment, I am exporting Clip G as a MIDI file so it can be uploaded into a DAW and assigned to a different instrument. The purpose is to hear whether changing the playback instrument affects comfort or interpretability while keeping the same `Vp`-derived note timing and pitch sequence.

This side experiment does not change the main sonification pipeline. It still uses only `Vp`, keeps the same Clip G aggregation and sustained-note behavior, and does not add `Np`, `Tpr`, chords, reverb, data-driven loudness, or a selected final instrument. The MIDI velocity is constant and is not a scientific amplitude mapping.

In [ ]:
from IPython.display import FileLink


def write_variable_length_quantity(value):
    """Encode an integer as a MIDI variable-length quantity."""
    value = int(value)
    if value < 0:
        raise ValueError("MIDI delta times must be non-negative")

    bytes_out = [value & 0x7F]
    value >>= 7
    while value:
        bytes_out.insert(0, (value & 0x7F) | 0x80)
        value >>= 7
    return bytes(bytes_out)


def midi_event(delta_ticks, status_byte, data_bytes):
    """Create one MIDI channel event with a variable-length delta time."""
    return write_variable_length_quantity(delta_ticks) + bytes([status_byte]) + bytes(data_bytes)


def midi_meta_event(delta_ticks, meta_type, data_bytes):
    """Create one MIDI meta event with a variable-length delta time."""
    return (
        write_variable_length_quantity(delta_ticks)
        + bytes([0xFF, meta_type])
        + write_variable_length_quantity(len(data_bytes))
        + bytes(data_bytes)
    )


def write_clip_g_midi(event_rows, output_path, ticks_per_quarter=480, tempo_microseconds_per_quarter=500_000):
    """Export Clip G sustained events as a simple type-0 MIDI file.

    Timing follows Clip G's 60-second audio timeline. Velocity is constant so the
    MIDI file does not introduce data-driven loudness.
    """
    ticks_per_second = ticks_per_quarter * 1_000_000 / tempo_microseconds_per_quarter
    midi_events = []
    velocity = 72
    channel = 0

    for event in event_rows:
        midi_note = event["midi_note"]
        if midi_note == "" or not np.isfinite(float(midi_note)):
            continue

        start_tick = int(round(event["audio_start_seconds"] * ticks_per_second))
        end_tick = int(round(event["audio_end_seconds"] * ticks_per_second))
        if end_tick <= start_tick:
            continue

        note = int(midi_note)
        midi_events.append((start_tick, 0, 0x90 | channel, [note, velocity]))
        midi_events.append((end_tick, 1, 0x80 | channel, [note, 0]))

    midi_events.sort(key=lambda item: (item[0], item[1]))

    track = bytearray()
    track += midi_meta_event(0, 0x03, b"HelioSonic Clip G - Vp")
    track += midi_meta_event(
        0,
        0x51,
        [
            (tempo_microseconds_per_quarter >> 16) & 0xFF,
            (tempo_microseconds_per_quarter >> 8) & 0xFF,
            tempo_microseconds_per_quarter & 0xFF,
        ],
    )
    track += midi_event(0, 0xC0 | channel, [0])

    previous_tick = 0
    for absolute_tick, _, status_byte, data_bytes in midi_events:
        delta = absolute_tick - previous_tick
        track += midi_event(delta, status_byte, data_bytes)
        previous_tick = absolute_tick

    track += midi_meta_event(0, 0x2F, [])

    header = b"MThd" + (6).to_bytes(4, "big") + (0).to_bytes(2, "big") + (1).to_bytes(2, "big") + ticks_per_quarter.to_bytes(2, "big")
    track_chunk = b"MTrk" + len(track).to_bytes(4, "big") + bytes(track)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_bytes(header + track_chunk)
    return output_path


clip_g_midi_path = write_clip_g_midi(
    narrow_event_tables["G"],
    Path("../media/clip_g_vp_sustained_midi_64_70.mid"),
)

clip_g_event_count = len(narrow_event_tables["G"])
print(f"Exported Clip G MIDI events: {clip_g_event_count}")
print(f"MIDI file: {clip_g_midi_path.resolve()}")
display(FileLink(str(clip_g_midi_path)))

## Clip G Experimentation in DAW Thoughts

I think that it did help, but not in the way that I hoped. Hearing the notes in as strings is much more pleasant than what can be outputted through the original clips. However, there are still huge gaps and jumps from note to note that make it sound jarring. I'm wondering if there is a way where say, if the same note lasts for 2 seconds, there is a 1/4th transition period where the note will slide in pitch from one note to the next, rather than jump. This also presents it's own problems, since if there is a high note that only lasts for 1/8th of a second that will prevent the transition from working. This is something that we need to figure out in the future. 